# 📦 Cenário de Dados: Sistema de Despachos

Este cenário simula o banco de dados de um aplicativo de logística e despachos. Os dados são altamente transacionais (mudanças constantes de status de motoristas e corridas), justificando o uso de tecnologias Lakehouse como o Delta Lake para garantir transações ACID.

## Modelo Entidade-Relacionamento (ER)
Abaixo, o relacionamento entre os motoristas cadastrados e as corridas realizadas.

```mermaid
erDiagram
    MOTORISTAS ||--o{ CORRIDAS : "realiza"
    
    MOTORISTAS {
        int id_motorista PK
        string nome
        string veiculo
        string status
    }
    
    CORRIDAS {
        int id_corrida PK
        int id_motorista FK
        string origem
        string destino
        float valor_corrida
        string status_corrida
    }

In [1]:
import os
import urllib.request
import pyspark
from delta import *

# =====================================================================
# 🛠️ CONSERTO PARA WINDOWS: Baixando o "tradutor" do Spark (Hadoop)
# =====================================================================
hadoop_home = os.path.abspath("hadoop_win")
hadoop_bin = os.path.join(hadoop_home, "bin")
os.makedirs(hadoop_bin, exist_ok=True)

base_url = "https://raw.githubusercontent.com/cdarlint/winutils/master/hadoop-3.2.0/bin/"

# Baixa os arquivos winutils.exe e hadoop.dll se você ainda não tiver
for file in ["winutils.exe", "hadoop.dll"]:
    file_path = os.path.join(hadoop_bin, file)
    if not os.path.exists(file_path):
        print(f"Baixando {file} para o Windows parar de reclamar...")
        urllib.request.urlretrieve(base_url + file, file_path)

# Avisa o Windows e o Spark onde estão os arquivos
os.environ["HADOOP_HOME"] = hadoop_home
os.environ["PATH"] += os.pathsep + hadoop_bin
# =====================================================================


# 1. Configurando o Spark para usar o Delta Lake
print("Ligando o motor do Apache Spark...")
builder = pyspark.sql.SparkSession.builder.appName("DeltaLake_Despachos") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR") # Esconde os avisos vermelhos gigantes

print("✅ Spark com Delta Lake iniciado com sucesso!")

# 2. Comandos DDL: Criando as Tabelas
spark.sql("""
CREATE TABLE IF NOT EXISTS motoristas (
    id_motorista INT,
    nome STRING,
    veiculo STRING,
    status STRING
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS corridas (
    id_corrida INT,
    id_motorista INT,
    origem STRING,
    destino STRING,
    valor_corrida FLOAT,
    status_corrida STRING
) USING DELTA
""")

print("✅ Tabelas 'motoristas' e 'corridas' criadas no formato Delta!")

Ligando o motor do Apache Spark...
✅ Spark com Delta Lake iniciado com sucesso!
✅ Tabelas 'motoristas' e 'corridas' criadas no formato Delta!


In [ ]:
# =====================================================================
# 3. Operações: INSERT, UPDATE e DELETE (Exigência do Professor)
# =====================================================================
print("🚀 Iniciando operações no Delta Lake...\n")

# --- INSERT ---
print("1️⃣ INSERT: Adicionando motoristas e corridas...")
spark.sql("""
INSERT INTO motoristas VALUES 
(1, 'João Silva', 'Fiat Uno', 'Ativo'),
(2, 'Maria Souza', 'Chevrolet Onix', 'Inativo')
""")

spark.sql("""
INSERT INTO corridas VALUES 
(101, 1, 'Centro', 'Aeroporto', 45.50, 'Em Andamento')
""")

print("Tabela Motoristas após o INSERT:")
spark.sql("SELECT * FROM motoristas").show()

# --- UPDATE ---
print("2️⃣ UPDATE: Atualizando o status da corrida para 'Finalizada'...")
spark.sql("""
UPDATE corridas 
SET status_corrida = 'Finalizada', valor_corrida = 50.00
WHERE id_corrida = 101
""")

print("Tabela Corridas após o UPDATE:")
spark.sql("SELECT * FROM corridas").show()

# --- DELETE ---
print("3️⃣ DELETE: Removendo motoristas inativos do sistema...")
spark.sql("""
DELETE FROM motoristas 
WHERE status = 'Inativo'
""")

print("Tabela Motoristas após o DELETE:")
spark.sql("SELECT * FROM motoristas").show()

🚀 Iniciando operações no Delta Lake...

1️⃣ INSERT: Adicionando motoristas e corridas...
Tabela Motoristas após o INSERT:
+------------+-----------+--------------+-------+
|id_motorista|       nome|       veiculo| status|
+------------+-----------+--------------+-------+
|           2|Maria Souza|Chevrolet Onix|Inativo|
|           1| João Silva|      Fiat Uno|  Ativo|
+------------+-----------+--------------+-------+

2️⃣ UPDATE: Atualizando o status da corrida para 'Finalizada'...
Tabela Corridas após o UPDATE:
+----------+------------+------+---------+-------------+--------------+
|id_corrida|id_motorista|origem|  destino|valor_corrida|status_corrida|
+----------+------------+------+---------+-------------+--------------+
|       101|           1|Centro|Aeroporto|         50.0|    Finalizada|
+----------+------------+------+---------+-------------+--------------+

3️⃣ DELETE: Removendo motoristas inativos do sistema...
Tabela Motoristas após o DELETE:
+------------+----------+------